# Suite CONF — Conformance across implementations

Every implementation writes the same files for the conformance corpus (`conformance/`): each language's grammar,
and for each source the tree it parses to, as a JSON snapshot, and the text that tree prints to. This suite checks
that this implementation's files are current, that they are byte-identical to every other implementation's, and
that every implementation's snapshots read back into trees that print the same text.

In [ ]:
import { readdirSync, readFileSync, statSync } from "node:fs";
import { join } from "node:path";

import { Ccpp20, Syntax as S } from "@mbse/programs/Ccpp";
import { render } from "@mbse/programs/Conformance/write";
import { Python314, Syntax as PS } from "@mbse/programs/Python";
import { Syntax as F } from "@mbse/programs/Framework";
import { JSON as SchemaJSON } from "@mbse/schemas/Framework";

import { assert, equal } from "./support.js";

const ROOT = "../../conformance";
const MINE = join(ROOT, "typescript5");
const OTHERS = readdirSync(ROOT).sort().filter((d) => statSync(join(ROOT, d)).isDirectory() && !["sources", "typescript5"].includes(d))
  .map((d) => join(ROOT, d));
const files = render(join(ROOT, "sources"));
const read = (path: string) => readFileSync(path, "utf8");

## CONF-01 · This implementation's files are current

In [ ]:
equal(readdirSync(MINE).sort(), [...files.keys()].sort());
for (const [name, text] of files) assert(read(join(MINE, name)) === text, `${name} is stale: run npm run conformance`);
for (const language of [S.LANGUAGE, PS.LANGUAGE]) {
  const grammar = JSON.parse(files.get(`${language.name()}.grammar.json`) as string);
  equal(grammar.kinds.map((k: any) => k.kind), [...language.kinds().keys()]);
}
console.log([...files.keys()].sort());

## CONF-02 · Every other implementation wrote the same bytes

In [ ]:
equal(OTHERS.map((d) => d.split("/").at(-1)), ["python3"]);
for (const other of OTHERS) {
  equal(readdirSync(other).sort(), [...files.keys()].sort());
  for (const [name, text] of files) assert(read(join(other, name)) === text, `${other}/${name} differs`);
}
console.log("identical to", OTHERS);

## CONF-03 · Every snapshot reads back into a valid tree that prints the same text

In [ ]:
const LANGUAGES = new Map<string, [{ print(node: F.Node): string; parse(text: string): F.Node }, F.Language, any]>([
  [".cpp", [Ccpp20, S.LANGUAGE, S.TranslationUnit]], [".py", [Python314, PS.LANGUAGE, PS.Module]]]);
for (const directory of [MINE, ...OTHERS]) {
  for (const name of readdirSync(directory).sort()) {
    const suffix = name.slice(name.lastIndexOf("."));
    const found = LANGUAGES.get(suffix);
    if (found === undefined) continue;
    const [standard, language, root] = found;
    const snapshot = read(join(directory, name.slice(0, -suffix.length) + ".json"));
    const unit = SchemaJSON.FromJSON(language.Builders).Reachable(root.Schema, snapshot) as F.Node;
    equal(language.validate(unit), []);
    const printed = read(join(directory, name));
    assert(standard.print(unit) === printed && F.same(standard.parse(printed), unit));
  }
}
console.log("ok");